In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

recipes = pd.read_csv(os.path.join(path, "RAW_recipes.csv"))
interactions = pd.read_csv(os.path.join(path, "RAW_interactions.csv"))

In [ ]:
recipes.head()

## Data cleaning

Our network links recipes that share ingredients, so the quality of the ingredient lists decides the quality of the edges. We clean the data in five steps and record how much each step removes, so every choice can be reported and defended:

1. Turn the text columns into real Python lists.
2. Standardise the ingredient names, so that variants like `eggs` and `egg` count as the same ingredient.
3. Remove recipes with too few ingredients.
4. Remove duplicate recipes.
5. Fix obviously wrong values in `minutes` and add a `year` column.

In [ ]:
import ast
from collections import Counter

n_start = len(recipes)
print(f"Recipes before cleaning: {n_start:,}")

### 1. Text columns to lists

In the CSV file, columns like `ingredients` look like lists but are stored as strings, e.g. `"['egg', 'flour']"`. `ast.literal_eval` turns each string into a real Python list, so we can loop over the ingredients.

In [ ]:
for col in ["ingredients", "tags", "nutrition", "steps"]:
    recipes[col] = recipes[col].apply(ast.literal_eval)

recipes["ingredients"].head()

### 2. Standardise ingredient names

The raw data has almost 15,000 different ingredient names, and many of them are the same ingredient written in different ways (`eggs`/`egg`, `garlic cloves`/`garlic`, `salt & pepper`). If we leave them as they are, two recipes that both use eggs will not share an ingredient in the network.

We use three simple rules:

- **Lowercase and remove extra spaces.**
- **Synonyms:** a small hand-made dictionary maps common variants to one name. Combined ingredients like `salt and pepper` are split into `salt` and `pepper`. The dictionary can be extended when we find more cases.
- **Plural to singular:** a trailing `s` is removed, but *only* if the singular form also exists as an ingredient in the data. This turns `eggs` into `egg` and `onions` into `onion`, while words like `molasses` stay unchanged, because `molasse` does not exist in the data.

We deliberately avoid more aggressive matching, which would turn e.g. `eggplant` into `egg` or `peanut butter` into `butter`.

In [ ]:
SYNONYMS = {
    "garlic cloves": "garlic",
    "garlic clove": "garlic",
    "minced garlic cloves": "garlic",
    "fresh garlic": "garlic",
    "large eggs": "egg",
    "unsalted butter": "butter",
    "salted butter": "butter",
    "extra virgin olive oil": "olive oil",
    "virgin olive oil": "olive oil",
    "kosher salt": "salt",
    "sea salt": "salt",
    "black pepper": "pepper",
    "fresh ground black pepper": "pepper",
    "freshly ground black pepper": "pepper",
    "ground black pepper": "pepper",
    "all-purpose flour": "flour",
    "granulated sugar": "sugar",
    "salt and pepper": "salt; pepper",
    "salt & pepper": "salt; pepper",
    "salt & freshly ground black pepper": "salt; pepper",
}

# All ingredient names in the data (lowercased), used by the plural rule
all_names = set(x.strip().lower() for xs in recipes["ingredients"] for x in xs)


def clean_ingredient(x):
    x = " ".join(x.lower().split())               # lowercase, remove extra spaces
    x = SYNONYMS.get(x, x)
    names = []
    for part in x.split("; "):                     # split combined ingredients
        if part.endswith("s") and part[:-1] in all_names:
            part = part[:-1]                       # plural -> singular
        names.append(part)
    return names


def clean_list(xs):
    cleaned = [name for x in xs for name in clean_ingredient(x)]
    return sorted(set(cleaned))                    # remove duplicates within a recipe


recipes["ingr_clean"] = recipes["ingredients"].apply(clean_list)
recipes["n_ingr_clean"] = recipes["ingr_clean"].str.len()

n_before = len(all_names)
n_after = recipes["ingr_clean"].explode().nunique()
print(f"Unique ingredient names: {n_before:,} -> {n_after:,}")

A quick check that the merging did what we wanted: the most common ingredients after cleaning.

In [ ]:
Counter(recipes["ingr_clean"].explode()).most_common(20)

### 3. Remove recipes with too few ingredients

An edge requires that two recipes share at least `MIN_SHARED = 2` ingredients. A recipe with fewer than 3 ingredients can hardly meet that, and will almost always become an isolated node. Such recipes are often not real dishes either (e.g. a drink or a spice mix), so we remove them.

In [ ]:
MIN_INGREDIENTS = 3

n_before = len(recipes)
recipes = recipes[recipes["n_ingr_clean"] >= MIN_INGREDIENTS]
print(f"Removed {n_before - len(recipes):,} recipes with fewer than {MIN_INGREDIENTS} ingredients")

### 4. Remove duplicate recipes

Some recipes have exactly the same ingredients as another recipe, often because the same recipe was uploaded twice. Duplicates create artificially strong edges (Jaccard similarity = 1), so we keep only the first recipe for each ingredient set.

In [ ]:
n_before = len(recipes)
recipes = recipes[~recipes["ingr_clean"].apply(tuple).duplicated()]
print(f"Removed {n_before - len(recipes):,} recipes with an identical ingredient list")

### 5. Fix `minutes` and add `year`

`minutes` contains impossible values: recipes that take 0 minutes, and one that takes over two billion minutes. We set values of 0 or above one day (1,440 minutes) to missing (`NaN`) instead of removing the recipes, because their ingredients are still fine.

We also add a `year` column from `submitted`. Note that most recipes were submitted between 2002 and 2009, and only a few hundred per year after 2014, which matters if we look at trends over time.

In [ ]:
bad_minutes = (recipes["minutes"] == 0) | (recipes["minutes"] > 24 * 60)
recipes.loc[bad_minutes, "minutes"] = None
print(f"Set {bad_minutes.sum():,} unrealistic cooking times to NaN")

recipes["year"] = recipes["submitted"].str[:4].astype(int)
recipes["year"].value_counts().sort_index()

### Summary

In [ ]:
recipes = recipes.reset_index(drop=True)

print(f"Recipes: {n_start:,} -> {len(recipes):,} ({len(recipes) / n_start:.1%} kept)")
print(f"Unique ingredients: {recipes['ingr_clean'].explode().nunique():,}")
print(f"Median number of ingredients per recipe: {recipes['n_ingr_clean'].median():.0f}")

## The recipe network

Nodes are recipes. Two recipes are linked if they share at least `MIN_SHARED` ingredients **and** their Jaccard similarity (shared ingredients / all ingredients in the two recipes) is at least `MIN_JACCARD`.

Ingredients that appear in more than 10 % of all recipes (salt, butter, sugar, ...) are ignored when we make the links, because sharing salt says nothing about two recipes being similar. We start with a random sample of 20,000 recipes to keep it fast.

To find the shared ingredients for all pairs at once, we build a recipe × ingredient matrix `X` with a 1 where a recipe uses an ingredient. Then `X @ X.T` gives the number of shared ingredients for every pair of recipes.

In [ ]:
import networkx as nx
from sklearn.preprocessing import MultiLabelBinarizer

SAMPLE_SIZE = 20_000
MAX_DOC_FREQ = 0.10
MIN_SHARED = 2
MIN_JACCARD = 0.2

sample = recipes.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Remove the too common ingredients (only for making the links)
doc_freq = sample["ingr_clean"].explode().value_counts() / len(sample)
too_common = set(doc_freq[doc_freq > MAX_DOC_FREQ].index)
edge_sets = sample["ingr_clean"].apply(lambda xs: [x for x in xs if x not in too_common])
print("Ignored ingredients:", sorted(too_common))

# Number of shared ingredients for every pair of recipes
X = MultiLabelBinarizer(sparse_output=True).fit_transform(edge_sets).tocsr()
S = (X @ X.T).tocoo()
size = edge_sets.str.len().to_numpy()

i, j, shared = S.row, S.col, S.data
jaccard = shared / (size[i] + size[j] - shared)
keep = (i < j) & (shared >= MIN_SHARED) & (jaccard >= MIN_JACCARD)   # i < j: each pair once

G = nx.Graph()
G.add_nodes_from(sample["id"])
ids = sample["id"].to_numpy()
G.add_weighted_edges_from(zip(ids[i[keep]], ids[j[keep]], jaccard[keep]))
print(f"{G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## Degree distribution

The degree of a recipe is the number of other recipes it is linked to. We plot how many recipes have each degree, on log–log axes (week 1): a heavy-tailed distribution looks roughly like a straight line there, while a distribution without hubs drops off steeply. Recipes with degree 0 can't be shown on a log axis, so we count them separately.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

degrees = np.array([d for _, d in G.degree()])
print(f"Average degree: {degrees.mean():.1f}, median: {np.median(degrees):.0f}, "
      f"max: {degrees.max()}, isolated recipes: {(degrees == 0).sum():,}")

counts = np.bincount(degrees)
k = np.arange(len(counts))

plt.loglog(k[1:], counts[1:], ".")
plt.xlabel("Degree $k$")
plt.ylabel("Number of recipes")
plt.title("Degree distribution of the recipe network")
plt.show()

## Drawing the whole network

Now we draw the full network `G` from above: all 20,000 recipes in the sample and all their links. With this many nodes the drawing must be kept simple: small nodes, thin and almost transparent edges, and no labels. We draw the largest connected component, which holds about 90 % of the recipes. Isolated recipes and small loose pieces are left out, because a force layout pushes them far out and squeezes the rest into a small blob.

NetworkX's own `spring_layout` takes more than 10 minutes on a network this size, so we compute the layout with `igraph` instead (install with `pip install igraph`). It uses the same idea, a force layout (Fruchterman–Reingold), but is written in C and takes about a minute. The drawing itself is still done with NetworkX.

In [ ]:
import igraph as ig

G_draw = G.subgraph(max(nx.connected_components(G), key=len))   # largest connected component
nodes = list(G_draw)

# Force layout with igraph (fast), then back to a NetworkX-style dict {node: (x, y)}
g = ig.Graph.from_networkx(G_draw)
layout = g.layout_fruchterman_reingold(grid=True)
pos = dict(zip(nodes, layout.coords))

plt.figure(figsize=(12, 12))
nx.draw(G_draw, pos, node_size=1, width=0.05, alpha=0.3)
plt.title(f"Largest component of the recipe network: {G_draw.number_of_nodes():,} recipes, {G_draw.number_of_edges():,} links")
plt.show()